数据下载 数据清洗与筛选

从CoinGecko网站下载的数据列名不够规范，为统计方便，列名统一改为"date", "price", "market_cap", "volume"

筛选时间：2019-01-01至2024-12-31

每个csv筛选后将名字改为_FULL.csv

In [15]:
import pandas as pd
import glob
import os

os.chdir(r"C:\Users\liulian\COMPLICATED NETWORK")
print("当前工作目录：", os.getcwd())

files = glob.glob("*.csv")
print("找到的CSV文件：", files)

for f in files:
    if f.endswith("_FULL.csv"):
        print(f"跳过已有FULL文件: {f}")
        continue

    print(f"\n正在处理: {f}")

    df = pd.read_csv(f)
    print("原始列名：", df.columns.tolist())
#改列名
    df = df.rename(columns={
        "snapped_at": "date",
        "total_volume": "volume"
    })

#检查
    required_cols = ["date", "price", "market_cap", "volume"]
    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        print(f"跳过 {f}，缺少列：{missing_cols}")
        continue
#设置筛选日期
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    df = df.dropna(subset=["date"])

    start_date = "2019-01-01"
    end_date = "2024-12-31"

    df = df[(df["date"] >= start_date) & (df["date"] <= end_date)]
    
    df = df.drop_duplicates(subset=["date"]).sort_values("date")

#最终保留    
    df = df[["date", "price", "market_cap", "volume"]]

    
    base_name = os.path.splitext(f)[0]
    out_file = f"{base_name}_FULL.csv"
    df.to_csv(out_file, index=False, encoding="utf-8-sig")

    print(f"已导出: {out_file} | 行数: {df.shape[0]}")

当前工作目录： C:\Users\liulian\COMPLICATED NETWORK
找到的CSV文件： ['ada-usd-max.csv', 'ada-usd-max_FULL.csv', 'bch-usd-max.csv', 'bch-usd-max_FULL.csv', 'btc.csv', 'btc_FULL.csv', 'cro-usd-max.csv', 'doge-usd-max.csv', 'doge-usd-max_FULL.csv', 'eth-usd-max.csv', 'eth-usd-max_FULL.csv', 'link-usd-max.csv', 'link-usd-max_FULL.csv', 'ltc-usd-max.csv', 'ltc-usd-max_FULL.csv', 'pyusd-usd-max.csv', 'ton-usd-max.csv', 'trx-usd-max.csv', 'trx-usd-max_FULL.csv', 'usdc-usd-max.csv', 'usdc-usd-max_FULL.csv', 'usdt-usd-max.csv', 'usdt-usd-max_FULL.csv', 'USYC-usd-max.csv', 'xlm-usd-max.csv', 'xlm-usd-max_FULL.csv', 'xmr-usd-max.csv', 'xmr-usd-max_FULL.csv', 'xrp-usd-max.csv', 'xrp-usd-max_FULL.csv', 'zec-usd-max.csv', 'zec-usd-max_FULL.csv']

正在处理: ada-usd-max.csv
原始列名： ['snapped_at', 'price', 'market_cap', 'total_volume']
已导出: ada-usd-max_FULL.csv | 行数: 2192
跳过已有FULL文件: ada-usd-max_FULL.csv

正在处理: bch-usd-max.csv
原始列名： ['snapped_at', 'price', 'market_cap', 'total_volume']
已导出: bch-usd-max_FULL.csv | 行数: 219

在网站CoinGecko中，按照市值(market cap)从高到低排序，选取前35个币
其中有五个币在所选时间范围期间没有数据
在余下的30个币的数据中，经过数据清洗和筛选。最终我们留下15个币，这15个币满足：

1.市场代表性

2.时间完整性：我们选取2019-01-01至2024-12-31的数据，共2191条数据，数量相差较大的币已被剔除（2191条数据的xmr和usdt仍保留）

样本数据完整性表

In [16]:
files = glob.glob("*_FULL.csv")
row_info = []
for f in files:
    df = pd.read_csv(f)
    row_info.append({
        "file": f,
        "rows": len(df),
        "start_date": df["date"].min(),
        "end_date": df["date"].max()
    })
row_df = pd.DataFrame(row_info).sort_values("rows", ascending=False)
print(row_df)

                      file  rows                 start_date  \
0     ada-usd-max_FULL.csv  2192  2019-01-01 00:00:00+00:00   
7     ltc-usd-max_FULL.csv  2192  2019-01-01 00:00:00+00:00   
16    xrp-usd-max_FULL.csv  2192  2019-01-01 00:00:00+00:00   
14    xlm-usd-max_FULL.csv  2192  2019-01-01 00:00:00+00:00   
11   usdc-usd-max_FULL.csv  2192  2019-01-01 00:00:00+00:00   
10    trx-usd-max_FULL.csv  2192  2019-01-01 00:00:00+00:00   
1     bch-usd-max_FULL.csv  2192  2019-01-01 00:00:00+00:00   
17    zec-usd-max_FULL.csv  2192  2019-01-01 00:00:00+00:00   
6    link-usd-max_FULL.csv  2192  2019-01-01 00:00:00+00:00   
5     eth-usd-max_FULL.csv  2192  2019-01-01 00:00:00+00:00   
4    doge-usd-max_FULL.csv  2192  2019-01-01 00:00:00+00:00   
2             btc_FULL.csv  2192  2019-01-01 00:00:00+00:00   
12   usdt-usd-max_FULL.csv  2191  2019-01-01 00:00:00+00:00   
3     cro-usd-max_FULL.csv  2191  2019-01-02 00:00:00+00:00   
15    xmr-usd-max_FULL.csv  2191  2019-01-01 00:00:00+0

删去明显不完整的

In [17]:
import os
import glob
import pandas as pd

os.chdir(r"C:\Users\liulian\COMPLICATED NETWORK")

full_files = glob.glob("*_FULL.csv")

deleted = []
not_found = []

for f in full_files:
    df = pd.read_csv(f)
    rows = len(df)
#删去明显不完整的：远少于2192条数据的币
    if rows < 2191:
        raw_file = f.replace("_FULL.csv", ".csv")
        full_file = f

        if os.path.exists(raw_file):
            os.remove(raw_file)
            deleted.append(raw_file)
        else:
            not_found.append(raw_file)

        if os.path.exists(full_file):
            os.remove(full_file)
            deleted.append(full_file)
        else:
            not_found.append(full_file)

print("\n已删除文件：")
for file in deleted:
    print(file)

print("\n未找到文件：")
for file in not_found:
    print(file)


已删除文件：
pyusd-usd-max.csv
pyusd-usd-max_FULL.csv
ton-usd-max.csv
ton-usd-max_FULL.csv
USYC-usd-max.csv
USYC-usd-max_FULL.csv

未找到文件：


In [18]:
files = sorted(glob.glob("*_FULL.csv"))
row_info = []
for f in files:
    df = pd.read_csv(f)
    row_info.append({
        "file": f,
        "rows": len(df)
    })
row_df = pd.DataFrame(row_info).sort_values("rows", ascending=False).reset_index(drop=True)
print("剩余币种数量：", len(row_df))
print(row_df)

剩余币种数量： 15
                     file  rows
0    ada-usd-max_FULL.csv  2192
1    bch-usd-max_FULL.csv  2192
2            btc_FULL.csv  2192
3   doge-usd-max_FULL.csv  2192
4    eth-usd-max_FULL.csv  2192
5   link-usd-max_FULL.csv  2192
6    ltc-usd-max_FULL.csv  2192
7    trx-usd-max_FULL.csv  2192
8   usdc-usd-max_FULL.csv  2192
9    xlm-usd-max_FULL.csv  2192
10   xrp-usd-max_FULL.csv  2192
11   zec-usd-max_FULL.csv  2192
12   cro-usd-max_FULL.csv  2191
13  usdt-usd-max_FULL.csv  2191
14   xmr-usd-max_FULL.csv  2191
